# Remote model interface

**Dr. Panagiotis Sakagiannis, Dr. Alexandros Marantis**

## About this notebook

The previous lesson replaced a brain module with a Python class. This one replaces it with a
**different process** - so the module can be written in a framework Larvaworld knows nothing about.
In the worked example that framework is [brian2](https://brian2.readthedocs.io), and the module is
a conductance-based leaky integrate-and-fire olfactory sensory neuron with spike-frequency
adaptation.

The mechanism is a socket. On every timestep the Larvaworld module packs the sensed odor
concentration into a message, sends it to a server, waits, and uses the reply as its output. The
server runs the neuron model for that timestep and answers with its firing rate.

This notebook is a **guided reading of a working example**, not a runnable simulation : it needs
`brian2` and two processes running side by side. The complete code lives in
`examples/brian2-interface/` in the repository, and every excerpt below is quoted from it.

**What you will be able to do afterwards**

- Decide when a remote module is worth the complexity, and when it is not.
- Recognize the six files of the example and what each is responsible for.
- Understand the message contract between the two processes.
- Explain why an external, stateful model needs warm-up and state snapshotting.

**Prerequisites** : [Custom brain modules](custom_brain_modules.ipynb).

**Cost** : reading only. Running the example additionally needs `brian2` and two terminals.

## Section 1 : The idea

Two processes, talking over a socket :

```{mermaid}
sequenceDiagram
    participant S as Larvaworld simulation
    participant O as LocalOlfactor (brain module)
    participant C as ipc.Client
    participant V as Model server (server.py)
    participant B as brian2 model

    S->>O: update() at timestep t
    O->>O: read sensed concentration from self.input
    O->>C: BrianInterfaceMessage(sim_id, agent_id, step, params)
    C->>V: socket send
    V->>B: restore state, inject current, run for remote_dt
    B-->>V: firing rate, voltage, current, adaptation
    V->>B: snapshot state for the next step
    V-->>C: reply message with OSN_rate
    C-->>O: response
    O->>O: self.output = f(OSN_rate)
    O-->>S: output feeds the downstream modules
```

Two things are worth noticing in that diagram.

**The simulation blocks.** The module waits for the reply, so every timestep costs a round trip
plus however long the external model takes. This is the price of the approach, and it is the reason
to reach for it only when the external model is genuinely doing something Larvaworld cannot.

**The external model is stateful.** A neuron has a membrane potential and an adaptation current
that must persist from one timestep to the next. The server is therefore responsible for storing
and restoring that state - it is not implicit in the message exchange.

## Section 2 : Running the example

The example needs two processes, so two terminals :

```bash
# Terminal 1 - the model server
cd examples/brian2-interface
python server.py

# Terminal 2 - the Larvaworld experiment
cd examples/brian2-interface
python larvaworld_launcher.py
```

The server also has a self-test that needs no simulation at all, which is the quickest way to check
that the socket layer works before involving Larvaworld :

```bash
python server.py test
```

It sends five synthetic messages and prints the replies.

## Section 3 : The six files

| file | responsibility |
|---|---|
| `larvaworld_launcher.py` | the Larvaworld side : registers the custom module and launches the experiment |
| `sensors.py` | the custom brain modules, including `LocalOlfactor` which does the remote call |
| `server.py` | the model server : listens, unpacks messages, dispatches to the model, replies |
| `server_runnable.py` | the execution logic : warm-up, state restore, input conversion, run, snapshot |
| `model.py` | the brian2 model definition itself |
| `utils.py` | timing helpers used to profile the round trips |

The sections below walk through the first four; `model.py` is ordinary brian2 and `utils.py` is
plumbing.

### `larvaworld_launcher.py`

The entry point on the Larvaworld side. It follows exactly the pattern of the previous lesson -
register the class as a mode, point the model at that mode, run the experiment - with the single
difference that the registered class talks to a server.

```python
from larvaworld.lib.model import BrainModuleDB
from sensors import LocalOlfactor

# overwrite mode 'osn' to use our custom LocalOlfactor class
BrainModuleDB.BrainModuleModes.olfactor.osn = LocalOlfactor

expID = "chemorbit_OSN"
exp_conf = reg.conf.Exp.getID(expID)
exp_conf.env_params.food_params.source_units.Source.odor.spread = 0.01

larva_group = exp_conf.larva_groups
larva_group_id = larva_group.keylist[0]
mm = reg.conf.Model.getID(larva_group[larva_group_id].model)
mm.brain.olfactor.mode = "osn"   # use our implementation

erun = sim.ExpRun(
    experiment=expID,
    modelIDs=["navigator", "OSNnavigator"],
    N=2,
    duration=0.5,
)
erun.simulate()
```

### `sensors.py`

`LocalOlfactor` subclasses `OSNOlfactor`, which already knows how to reach a server - the base class
provides `self.brianInterface`, configured by the host, port, timestep and warm-up passed to
`super().__init__`.

Its `update()` does four things : build the payload from what the larva currently senses, execute a
remote step, convert the reply into an output, and remember the previous response so that a
*relative* change can be reported rather than an absolute rate.

```python
class LocalOlfactor(OSNOlfactor):
    def __init__(self, server_port=5795, remote_dt=100, remote_warmup=500, **kwargs):
        self.last_osn_activity = None
        super().__init__(
            response_key="OSN_rate",
            server_host="localhost",
            server_port=server_port,
            remote_dt=remote_dt,
            remote_warmup=remote_warmup,
            **kwargs,
        )

    def update(self):
        agent_id = self.brain.agent.unique_id
        sim_id = self.brain.agent.model.id

        msg_kws = {
            "odor_id": 0,
            "concentration_mmol": list(self.input.values())[0] * 220,
            "concentration_change_mmol": self.first_odor_concentration_change,
            "concentration_max_mmol": 440,     # mMol
            "baseline_input_current": 100,     # pA
            "g_Ia": 0,
            "max_rate": 100,
            "min_rate": 40,
        }

        response = self.brianInterface.executeRemoteModelStep(
            sim_id, agent_id, self.remote_dt, t_warmup=15000, **msg_kws
        )
        current = response.param(self.response_key)

        if self.last_osn_activity is None:
            self.last_osn_activity = current

        # report the relative change rather than the absolute rate
        self.output = ((current - self.last_osn_activity) / self.last_osn_activity) * 100
        self.last_osn_activity = current
```

Note the units in the payload : concentrations in mMol, currents in pA. The message carries plain
numbers, so the two sides have to agree on what those numbers mean - there is no unit checking
across the socket.

### `server.py`

The server side. `larvaworld.lib.ipc.Server` does the socket work; you supply a handler that
receives a batch of messages and returns a batch of replies.

```python
from larvaworld.lib.ipc import BrianInterfaceMessage, Server

def process_model(msg: BrianInterfaceMessage):
    odor_concentration = msg.param("concentration_mmol") or 450
    ...
    OSN_rate, OSN_voltage, OSN_current, OSN_adaptation = execute_model(
        id=msg.model_id,
        sim_id=msg.sim_id,
        step_id=msg.step,
        odor_concentration=odor_concentration,
        ...
    )
    return msg.with_params(
        OSN_rate=OSN_rate,
        OSN_voltage=OSN_voltage,
        OSN_current=OSN_current,
        OSN_adaptation=OSN_adaptation,
    )

def server_process_request(objects):
    with ThreadPoolExecutor(max_workers=N_threads) as executor:
        return executor.map(process_model, objects)

s = Server(server_address, server_process_request)
s.serve_forever()
```

Replies are built with `msg.with_params(...)`, which keeps the identifying fields - `sim_id`,
`model_id`, `step` - and swaps the payload. That is what lets one server serve many agents of many
simulations at once : the identity travels with every message, so nothing has to be tracked
per-connection.

### `server_runnable.py`

Where the two subtleties of a stateful external model are handled.

**Warm-up.** The first time a model instance runs, it is driven with a constant input for a while so
that membrane potential and adaptation settle into equilibrium. Without it the first few hundred
timesteps of the simulation would be reading a transient that has nothing to do with the odor.

**Snapshotting.** After every step the model state is saved, and before every step it is restored.
Skip this and the neuron behaves as if freshly initialized on every timestep - it would have no
adaptation, no memory of the previous concentration, and the whole point of using a spiking model
would be lost.

The file is heavily commented and prints what it is doing, which makes it a good skeleton to copy
for a different external model.

## Section 4 : The message

Everything crossing the socket is a `BrianInterfaceMessage`. It is deliberately minimal : three
identifying fields and an open dictionary of parameters.

| field | meaning |
|---|---|
| `sim_id` | which simulation the message belongs to |
| `model_id` | which agent's model instance, so state is kept per animal |
| `step` | the simulation timestep |
| `params` | everything else, as keyword arguments |

The cell below builds one and reads it back. It needs no server and no `brian2` - it is here so you
can see the object rather than infer it from the code above.

In [1]:
%matplotlib inline

from larvaworld.lib.ipc import BrianInterfaceMessage

msg = BrianInterfaceMessage(
    sim_id="demo_sim",
    model_id="larva_0",
    step=42,
    odor_id=0,
    concentration_mmol=180.0,
    baseline_input_current=100,
)

print(f"sim_id   : {msg.sim_id}")
print(f"model_id : {msg.model_id}")
print(f"step     : {msg.step}")
print(f"params   : {msg.params}")
print()

# the reply keeps the identity and swaps the payload
reply = msg.with_params(OSN_rate=63.5)
print(f"reply    : step={reply.step} OSN_rate={reply.param('OSN_rate')}")
print(f"missing parameters come back as None : {reply.param('not_there')}")

sim_id   : demo_sim
model_id : larva_0
step     : 42
params   : {'odor_id': 0, 'concentration_mmol': 180.0, 'baseline_input_current': 100}

reply    : step=42 OSN_rate=63.5
missing parameters come back as None : None


## Section 5 : When to use this

The remote interface costs a socket round trip per timestep per agent. That is worth paying when :

- the model you want already exists in another framework and porting it would be a project of its
  own,
- the model needs a different numerical integrator or a much smaller internal timestep than the
  behavioral simulation,
- the model is developed by someone else and you want to keep it as its own artifact.

It is not worth paying when the computation can be expressed as a Python function of the module's
input - in that case the previous lesson is the right tool, and the whole simulation stays in one
process.

## Where to go next

- [Custom brain modules](custom_brain_modules.ipynb) - the in-process alternative.
- `examples/brian2-interface/` in the repository - the complete, working code quoted above.
- Reference : [Brain module architecture](../../agents_environments/brain_module_architecture.md).